# 第 49 章：Capstone 公开发布 QA 与课程采用说明

这个 notebook 用一个极小的 launch-QA table，对比“只按发布优先级排序”的 naive baseline 和“先检查链接、发布 smoke test、支持边界与采用说明”的公开发布 QA workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_launch_qa_adoption_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['launch_priority_score'] = float(row['launch_priority_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} launch-QA items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Adopter role:', ordered_counts(rows, 'adopter_role'))
print('Link check:', ordered_counts(rows, 'link_check_status'))
print('Smoke test:', ordered_counts(rows, 'smoke_test_status'))
print('Adoption note:', ordered_counts(rows, 'adoption_note_status'))
print('Support boundary:', ordered_counts(rows, 'support_boundary_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['qa_id']
    for row in rows
    if row['reference_route'] == 'launch_ready'
)
top4_priority = sorted(
    rows,
    key=lambda row: row['launch_priority_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'launch_ready'
    for row in top4_priority
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'launch_ready'
    for row in top4_priority
)
baseline_ready_precision = baseline_ready_hits / len(top4_priority)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_priority)

print('Reference launch-ready items:', reference_ready_ids)
print('Top-4 by launch priority:')
for row in top4_priority:
    print(
        f"  {row['qa_id']}: priority={row['launch_priority_score']:.2f}, "
        f"route={row['reference_route']}, area={row['resource_area']}"
    )
print('Baseline launch precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_launch_item(row):
    if row['link_check_status'] == 'broken':
        return 'fix_broken_link', 'entry point, path, or linked artifact is broken'
    if row['smoke_test_status'] in {'failing', 'missing'}:
        return 'rerun_release_smoke_test', 'release smoke test is failing or missing'
    if row['support_boundary_status'] != 'clear':
        return 'clarify_support_boundary', 'support boundary, audience, or response promise is unclear'
    if row['adoption_note_status'] != 'complete':
        return 'complete_adoption_note', 'external adoption note is not complete enough'
    return 'launch_ready', 'links work, tests pass, boundary is clear, and adoption note is complete'


routed_rows = []
for row in rows:
    route, reason = route_launch_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Launch-QA workflow routes:')
for row in routed_rows:
    print(
        f"  {row['qa_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'launch_ready']
link_queue = [row for row in routed_rows if row['route'] == 'fix_broken_link']
smoke_queue = [row for row in routed_rows if row['route'] == 'rerun_release_smoke_test']
note_queue = [row for row in routed_rows if row['route'] == 'complete_adoption_note']
boundary_queue = [row for row in routed_rows if row['route'] == 'clarify_support_boundary']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'launch_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Launch-ready queue:', [row['qa_id'] for row in ready_queue])
print('Fix-link queue:', [row['qa_id'] for row in link_queue])
print('Rerun-smoke-test queue:', [row['qa_id'] for row in smoke_queue])
print('Complete-adoption-note queue:', [row['qa_id'] for row in note_queue])
print('Clarify-support-boundary queue:', [row['qa_id'] for row in boundary_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured launch precision:', round(ready_precision, 3))


In [ ]:
def launch_qa_steps(row):
    steps = []
    if row['link_check_status'] == 'broken':
        steps.append('先修复入口、相对路径、网页链接或下载链接，并记录检查日期。')
    if row['smoke_test_status'] in {'failing', 'missing'}:
        steps.append('在干净环境重跑发布 smoke test，记录 Python 版本、依赖和失败日志。')
    if row['support_boundary_status'] != 'clear':
        steps.append('写清支持边界：接受哪些 issue、哪些请求不承诺响应。')
    if row['adoption_note_status'] != 'complete':
        steps.append('补采用说明：适用对象、最小采用路径、运行时间和本地化建议。')
    return steps or ['可以进入 launch package；发布时保留版本号、检查日期和 known caveats。']


for row in routed_rows:
    if row['route'] != 'launch_ready':
        print(f"\n{row['qa_id']} -> {row['route']} ({row['resource_area']})")
        for step in launch_qa_steps(row):
            print(' -', step)


In [ ]:
adoption_note_outline = [
    'Audience and prerequisites: student level, prior Python, and science background',
    'Adoption paths: single chapter, three-week module, half course, or full course',
    'Runtime and environment: dependencies, data paths, timing, and known failures',
    'Assessment boundary: reusable rubric pieces and parts requiring localization',
    'Support boundary: issue policy, response expectation, and non-supported requests',
    'Maintenance cadence: version, last check date, next check date, and stale-material trigger',
]

launch_qa_assistant_prompt = '''你是我的 capstone 公开发布 QA 与课程采用助手。

任务：
1. 先阅读 launch-QA table，不要只按 launch priority 排序；
2. 先检查 link check；
3. 再检查 release smoke test、support boundary 和 adoption note；
4. 把每个资源 route 到 launch_ready、fix_broken_link、
   rerun_release_smoke_test、complete_adoption_note 或 clarify_support_boundary；
5. 对每个非 ready 资源输出发布前 QA checklist。

输出格式：
- Launch-ready resources
- Fix-link resources
- Rerun-smoke-test resources
- Complete-adoption-note resources
- Clarify-support-boundary resources
- Launch QA checklist by resource
'''

print('Adoption note outline:')
for item in adoption_note_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(launch_qa_assistant_prompt)


In [ ]:
launch_snapshot = {
    'dataset': DATA_PATH.name,
    'n_launch_items': len(rows),
    'baseline_top4_launch_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'launch_ready': [row['qa_id'] for row in ready_queue],
    'fix_broken_link': [row['qa_id'] for row in link_queue],
    'rerun_release_smoke_test': [row['qa_id'] for row in smoke_queue],
    'complete_adoption_note': [row['qa_id'] for row in note_queue],
    'clarify_support_boundary': [row['qa_id'] for row in boundary_queue],
    'python_version': platform.python_version(),
}

print('Launch QA adoption snapshot:')
for key, value in launch_snapshot.items():
    print(f'  {key}: {value}')
